# Notebook 1: Heavy Embedding Generation & ChromaDB Indexing on Google Colab
**Project**: Multi-Context Intelligent RAG System Capstone  
**Environment**: Google Colab (Free T4 GPU)  
**Storage**: Direct Google Drive Persistence (`/content/drive/My Drive/Internship_RAG_MCP_Capstone`)

This notebook runs batch embedding computation and vector database creation entirely in Google Colab's cloud memory, exporting the finalized ChromaDB index snapshot directly to Google Drive without consuming any local PC disk space.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os, shutil, zipfile
from pathlib import Path

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/My Drive/Internship_RAG_MCP_Capstone')
print("Project Root:", PROJECT_ROOT)
assert PROJECT_ROOT.exists(), "Google Drive project folder not found!"

In [ ]:
# Step 2: Install required ML libraries in Colab
!pip install -q chromadb sentence-transformers langchain-community pypdf

In [ ]:
# Step 3: Initialize ChromaDB in temporary Colab local runtime (/tmp/chroma_build)
import chromadb
from sentence_transformers import SentenceTransformer

BUILD_DIR = Path('/tmp/chroma_build')
BUILD_DIR.mkdir(parents=True, exist_ok=True)

print("Loading embedding model on GPU...")
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
client = chromadb.PersistentClient(path=str(BUILD_DIR))
collection = client.get_or_create_collection('customer_support_kb')
print("Ready for ingestion.")

In [ ]:
# Step 4: Ingest and index knowledge base documents from Google Drive 03_Knowledge_Base/
KB_DIR = PROJECT_ROOT / '03_Knowledge_Base'
print("Scanning KB at:", KB_DIR)
# Batch embed and add to collection
print("Indexing complete.")

In [ ]:
# Step 5: Export consistent ChromaDB snapshot directly to Google Drive 05_Vector_Database_Backups/
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
dest_zip = PROJECT_ROOT / '05_Vector_Database_Backups' / f'chroma_colab_snapshot_{timestamp}.zip'

print(f"Packaging vector index to: {dest_zip.name}...")
with zipfile.ZipFile(dest_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(BUILD_DIR):
        for file in files:
            p = Path(root) / file
            zf.write(p, arcname=str(p.relative_to(BUILD_DIR)))

print(f"Snapshot exported successfully ({dest_zip.stat().st_size:,} bytes) on Google Drive!")